# Encoder

In [ ]:
import math
import time
from collections import Counter

with open('war_and_peace_cn.txt', 'r', encoding='utf-8') as f:
    content = f.read()
# print(content)
counter = Counter(content)
total = sum(counter.values())

symbols = list(counter.keys())
probs = [counter[s] / total for s in symbols]

# encoder
def shannon_code(symbols, probs):
    codes = {}
    q = 0.0
    
    # sort the probabilities(large to small)
    sorted_probs = sorted(zip(symbols, probs), key=lambda x:-x[1])
    
    for s, p in sorted_probs:
        l = math.ceil(-math.log2(p)) # get optimal codeword length
        code = ''
        q_bin = q
        # get binary value
        for _ in range(l):
            q_bin *= 2
            bit = int(q_bin)
            code += str(bit)
            q_bin -= bit
        codes[s] = code
        q += p
    return codes

codes = shannon_code(symbols, probs)

for s in symbols:
    if s == '\n':
        display_s = '\\n'
    elif s == ' ':
        display_s = "' '"
    else:
        display_s = s
    print(f"symbol: {display_s} code: {codes[s]}")
    
entropy = -sum(p * math.log2(p) for p in probs)

avg_code_len = sum([len(codes[s]) * probs[i] for i, s in enumerate(symbols)])

efficiency = entropy / avg_code_len if avg_code_len > 0 else 0

print(f"\ninfo source entropy: {entropy:.4f} bits")
print(f"avg. codeword length: {avg_code_len:.4f} bits/symbol")
print(f"encoding efficiency: {efficiency * 100:.2f}%")

# Decoder

In [ ]:
# encode the whole text
encoded_text = ''.join([codes[c] for c in content])
# print(encoded_text)

# build decoding mapping
decode_map = {v: k for k, v in codes.items()}

# decoder
def shannon_decode(encoded_str, decode_map):
    decoded = []
    buffer = ''
    max_code_len = max(len(code) for code in decode_map)
    i = 0
    while i < len(encoded_str):
        buffer = ''
        for l in range(1, max_code_len + 1):
            if i + l > len(encoded_str):
                break
            buffer = encoded_str[i:i+l]
            if buffer in decode_map:
                decoded.append(decode_map[buffer])
                i += l
                break
        else:
            # fails to match
            break
    return ''.join(decoded)

# decode and time 
start_time = time.time()
decoded_text = shannon_decode(encoded_text, decode_map)
end_time = time.time()

print(f"\nwhether decoding is correct: {decoded_text == content}")
print(f"decoding time : {end_time - start_time:.6f} seconds")
# print("\ndecoded text: ")
# print(decoded_text)

# README

This notebook implements Shannon coding for text compression and decompression.

---

## Encoder

The encoder performs the following steps:

1. **Read and Count Symbols**  
   The text file is read, and the frequency of each symbol (including characters, spaces, and punctuation) is counted using `collections.Counter`:
   ```python
   with open('war_and_peace.txt', 'r', encoding='utf-8') as f:
       content = f.read()
   counter = Counter(content)
   total = sum(counter.values())
   ```

2. **Calculate Probabilities**  
   The probability of each symbol is calculated:
   ```python
   symbols = list(counter.keys())
   probs = [counter[s] / total for s in symbols]
   ```

3. **Generate Shannon Codes**  
   The `shannon_code` function sorts symbols by probability and assigns each a binary code according to Shannon's method:
   ```python
   def shannon_code(symbols, probs):
       codes = {}
       q = 0.0
       sorted_probs = sorted(zip(symbols, probs), key=lambda x:-x[1])
       for s, p in sorted_probs:
           l = math.ceil(-math.log2(p))
           code = ''
           q_bin = q
           for _ in range(l):
               q_bin *= 2
               bit = int(q_bin)
               code += str(bit)
               q_bin -= bit
           codes[s] = code
           q += p
       return codes
   codes = shannon_code(symbols, probs)
   ```

4. **Display Codes and Statistics**  
   The code for each symbol is printed, and the source entropy, average codeword length, and encoding efficiency are calculated:
   ```python
   entropy = -sum(p * math.log2(p) for p in probs)
   avg_code_len = sum([len(codes[s]) * probs[i] for i, s in enumerate(symbols)])
   efficiency = entropy / avg_code_len if avg_code_len > 0 else 0
   print(f"\ninfo source entropy: {entropy:.4f} bits")
   print(f"avg. codeword length: {avg_code_len:.4f} bits/symbol")
   print(f"encoding efficiency: {efficiency * 100:.2f}%")
   ```

---

## Decoder

The decoder performs the following steps:

1. **Encode the Entire Text**  
   The original text is encoded into a bit string using the generated codes:
   ```python
   encoded_text = ''.join([codes[c] for c in content])
   ```

2. **Build Decoding Map**  
   A reverse mapping from codewords to symbols is created:
   ```python
   decode_map = {v: k for k, v in codes.items()}
   ```

3. **Decode the Bit String**  
   The `shannon_decode` function scans the encoded bit string and matches codewords to symbols:
   ```python
   def shannon_decode(encoded_str, decode_map):
       decoded = []
       buffer = ''
       max_code_len = max(len(code) for code in decode_map)
       i = 0
       while i < len(encoded_str):
           buffer = ''
           for l in range(1, max_code_len + 1):
               if i + l > len(encoded_str):
                   break
               buffer = encoded_str[i:i+l]
               if buffer in decode_map:
                   decoded.append(decode_map[buffer])
                   i += l
                   break
           else:
               break
       return ''.join(decoded)
   ```

4. **Verify and Output**  
   The decoded text is compared with the original to verify correctness, and the decoding time is measured:
   ```python
   start_time = time.time()
   decoded_text = shannon_decode(encoded_text, decode_map)
   end_time = time.time()
   print(f"\nwhether decoding is correct: {decoded_text == content}")
   print(f"decoding time : {end_time - start_time:.6f} seconds")
   ```

---

**Note:**  
- All symbols, including punctuation and whitespace, are encoded and decoded.
